# M05-P01 — Tokens, contexto y presupuesto de generación

En M01 ya vimos temperatura, `maxTokens`, contexto y streaming. Aquí no repetimos aquello.

La pregunta ahora es:

```text
¿qué entra realmente en el contexto del modelo
y cómo puedo medirlo?
```

Con Claude Haiku 4.5 el curso usa un perfil de inferencia geográfico en `bedrock-runtime`. Ese endpoint no admite `CountTokens` para este modelo, así que para el recuento previo utilizaremos el endpoint oficial `bedrock-mantle`, que sí expone la API de conteo de tokens para Claude.

El objetivo no es aprender otro SDK: es hacer observable el presupuesto de contexto.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from enterprise_genai.config import get_settings
from enterprise_genai.models.bedrock import bedrock_runtime
from enterprise_genai.token_counting import count_claude_tokens

settings = get_settings()
runtime = bedrock_runtime()
MODEL_ID = settings.bedrock_text_model_id

# Modelo actual del curso: Claude Haiku 4.5.
CONTEXT_WINDOW = 200_000
MAX_OUTPUT_TOKENS = 64_000

print("Modelo de inferencia:", MODEL_ID)
print("Región:", settings.aws_region)
print("Ventana usada en el ejercicio:", f"{CONTEXT_WINDOW:,}", "tokens")

## 1. Contar tokens antes de inferencia

Para Claude Haiku 4.5, el recuento previo se realiza mediante la API de conteo de Anthropic expuesta por Bedrock Mantle.

El helper del proyecto oculta únicamente el detalle de firma HTTP. Nosotros seguimos viendo lo importante: **mensajes, system prompt y número de tokens**.

In [ ]:
def user_message(text):
    return [{"role": "user", "content": text}]

print(
    "Tokens para una petición con «Hola»:",
    count_claude_tokens(user_message("Hola")),
)

## 2. Palabras, caracteres y tokens no son lo mismo

`split()` solo cuenta palabras: **no es un tokenizer**.

Comparamos lenguaje natural, identificadores, emoji, JSON y código. El objetivo es observar que el modelo segmenta el texto según su tokenizer, no según espacios o caracteres.

In [ ]:
examples = {
    "frase corta": "La VPN no funciona.",
    "frase técnica": "La autenticación SAML de la VPN devuelve HTTP 401 desde las 08:15.",
    "identificador": "INC-2026-10-18-ACME-VPN-004821",
    "emoji": "VPN caída 🔐🚨 — revisar autenticación",
    "json": '{"service":"vpn","status":"degraded","users":73}',
    "python": "if status_code == 401:\n    retry_authentication(user_id)",
}

rows = []
for label, text in examples.items():
    tokens = count_claude_tokens(user_message(text))
    rows.append({
        "ejemplo": label,
        "caracteres": len(text),
        "palabras": len(text.split()),
        "input_tokens": tokens,
        "chars/token": round(len(text) / tokens, 2),
    })

token_df = pd.DataFrame(rows)
token_df

## 3. El formato y el `system prompt` también consumen contexto

Ahora comparamos la misma información expresada de distintas maneras y medimos el efecto de las instrucciones de sistema.

In [ ]:
variants = {
    "conciso": "73 usuarios no pueden autenticarse en la VPN desde las 08:15.",
    "verboso": (
        "Se ha detectado una situación en la que un total de 73 usuarios de ACME "
        "no consiguen completar correctamente el proceso de autenticación requerido "
        "para acceder al servicio corporativo de VPN desde aproximadamente las 08:15."
    ),
    "json": json.dumps({
        "affected_users": 73,
        "service": "VPN",
        "problem": "authentication failure",
        "since": "08:15",
    }),
    "lista": "- usuarios: 73\n- servicio: VPN\n- problema: autenticación\n- inicio: 08:15",
}

variant_rows = []
for label, text in variants.items():
    variant_rows.append({
        "formato": label,
        "caracteres": len(text),
        "palabras": len(text.split()),
        "input_tokens": count_claude_tokens(user_message(text)),
    })

print("Mismo significado aproximado, distinto formato:")
display(pd.DataFrame(variant_rows).sort_values("input_tokens"))

QUESTION = "Resume el incidente en una frase."
SYSTEM_SHORT = "Eres un analista de operaciones."
SYSTEM_LONG = (
    "Eres un analista senior de operaciones empresariales. Distingue hechos observados de hipótesis, "
    "evita afirmar una causa raíz sin evidencia, señala incertidumbres y prioriza acciones de validación."
)

system_rows = []
for label, system_text in [
    ("sin system", None),
    ("system corto", SYSTEM_SHORT),
    ("system detallado", SYSTEM_LONG),
]:
    system_rows.append({
        "configuración": label,
        "input_tokens": count_claude_tokens(user_message(QUESTION), system_text),
    })

print("\nEfecto del system prompt:")
display(pd.DataFrame(system_rows))

## 4. Una conversación crece turno a turno

Para mantener contexto, una aplicación suele volver a enviar el historial relevante. Vamos a medir prefijos de una conversación.

In [ ]:
conversation = [
    {"role": "user", "content": "Tenemos 73 usuarios sin acceso VPN desde las 08:15."},
    {"role": "assistant", "content": "¿Está afectado también el acceso web corporativo?"},
    {"role": "user", "content": "No. La web corporativa funciona correctamente."},
    {"role": "assistant", "content": "¿Qué muestra el estado del backend de autenticación?"},
    {"role": "user", "content": "El dashboard marca VPN AUTH BACKEND como DEGRADED."},
    {"role": "assistant", "content": "Eso aporta correlación, pero todavía no confirma la causa raíz."},
    {"role": "user", "content": "¿Qué comprobarías a continuación?"},
]

history_rows = []
for end in [1, 3, 5, 7]:
    prefix = conversation[:end]
    history_rows.append({
        "mensajes": len(prefix),
        "input_tokens": count_claude_tokens(prefix, SYSTEM_SHORT),
    })

history_df = pd.DataFrame(history_rows)
display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(history_df["mensajes"], history_df["input_tokens"], marker="o")
plt.xlabel("Mensajes incluidos")
plt.ylabel("Input tokens")
plt.title("Crecimiento del contexto con el historial")
plt.grid(alpha=0.25)
plt.show()

## 5. Recuento previo frente al uso real

Medimos el contenido antes de inferencia y después ejecutamos la petición real con `Converse`.

Hay un matiz importante: el recuento previo usa la API Anthropic Messages de `bedrock-mantle`, mientras la aplicación usa `Converse` en `bedrock-runtime`. Por ello los números pueden no ser idénticos debido al envoltorio de cada API. Lo importante es observar el orden de magnitud y el crecimiento del contexto.

In [ ]:
INCIDENT = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. El equipo de operaciones está investigando."
)
SYSTEM = "Eres un analista de operaciones. Resume solo hechos disponibles y no inventes una causa raíz."
USER_TEXT = INCIDENT + "\n\nDevuelve un resumen de máximo tres viñetas."

predicted_tokens = count_claude_tokens(user_message(USER_TEXT), SYSTEM)

response = runtime.converse(
    modelId=MODEL_ID,
    system=[{"text": SYSTEM}],
    messages=[{"role": "user", "content": [{"text": USER_TEXT}]}],
    inferenceConfig={"temperature": 0, "maxTokens": 180},
)

print(response["output"]["message"]["content"][0]["text"])
print()
print("Conteo previo (Mantle Messages):", predicted_tokens)
print("usage.inputTokens (Runtime Converse):", response["usage"]["inputTokens"])
print("usage.outputTokens:", response["usage"]["outputTokens"])
print("usage.totalTokens:", response["usage"]["totalTokens"])
print("stopReason:", response["stopReason"])

## 6. Presupuesto de contexto y puente hacia RAG

Podemos razonar como una aplicación:

```text
ventana total - entrada - reserva de salida = margen disponible
```

Que un documento **quepa** no significa que enviarlo entero sea siempre la mejor estrategia: también importan coste, latencia y relevancia.

In [ ]:
def context_budget(input_tokens, reserve_output=2_000, context_window=CONTEXT_WINDOW):
    available = context_window - input_tokens - reserve_output
    return {
        "context_window": context_window,
        "input_tokens": input_tokens,
        "reserved_output_tokens": reserve_output,
        "available_tokens": max(0, available),
        "used_percent_before_output": round(
            100 * (input_tokens + reserve_output) / context_window, 3
        ),
        "fits": available >= 0,
    }

print("Presupuesto estimado de la petición anterior:")
print(context_budget(predicted_tokens))

paragraph = (
    "Procedimiento de acceso remoto. Los usuarios deben utilizar MFA para conectarse a la VPN. "
    "Los errores de autenticación deben correlacionarse con los logs del proveedor de identidad "
    "y con el estado del backend antes de declarar una causa raíz."
)

long_rows = []
for n in [1, 10, 50, 100, 250]:
    text = "\n\n".join([paragraph] * n)
    tokens = count_claude_tokens(user_message(text))
    long_rows.append({
        "bloques": n,
        "caracteres": len(text),
        "input_tokens": tokens,
        "% ventana": round(tokens / CONTEXT_WINDOW * 100, 3),
    })

display(pd.DataFrame(long_rows))

## 7. MODIFICA

Prueba una opción:

- añade dos turnos a la conversación;
- amplía el `system prompt`;
- añade una variante XML;
- reserva 8.000 tokens de salida;
- aumenta el documento hasta superar el 25 % de la ventana.

### Pregunta final

Si un documento completo cabe en contexto, ¿por qué podríamos preferir recuperar solo los fragmentos relevantes?

Guarda la pregunta: será central en embeddings y RAG.